In [ ]:
%config InlineBackend.figure_format = 'retina'
import os
import sys
import importlib
from pathlib import Path
root_path = Path(os.getcwd())
sys.path.append(str(Path('..').resolve()))

import math
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from guardians_2025 import TOP_100

from loader import load_player_data_for_scoring
from rosters_2026 import rosters_2026
from aggregation import aggregate_all
from steps.filter import filter_all, filtering_summary
from steps.decay import apply_decay, decay_summary
from steps.normalization import normalize_all, normalization_summary
from steps.segmentation import apply_segmentation, segmentation_summary
from steps.scoring import build_scores, scoring_summary
import steps.normalization
from steps.normalization import *

STATSBOMB_DIR = Path("..") / "data" / "Statsbomb"

In [21]:
# Build flat set of all rostered players
import rosters_2026
importlib.reload(rosters_2026)
from rosters_2026 import rosters_2026
rostered_players = {
    player 
    for squad in rosters_2026.values() 
    for player in squad.keys()
}
print(f"Total rostered players: {len(rostered_players)}")

Total rostered players: 664


In [22]:
matches = pl.read_parquet("../data/Statsbomb/matches.parquet")
raw = load_player_data_for_scoring("recent_club_players")
clean = aggregate_all(raw, matches, verbose=False)


Loading: recent_club_players
Seasons: ['2021/2022', '2022/2023', '2023/2024']

  ── 2021/2022 ──
    ✓ xg__player__totals.csv — 293 rows, 284 players
    ✓ progression__player__profile.csv — 1,175 rows, 1123 players
    ✓ advanced__player__packing.csv — 5,104 rows, 1053 players
    ✓ advanced__player__xg_chain.csv — 5,403 rows, 1151 players
    ✓ advanced__player__xg_buildup.csv — 4,731 rows, 1083 players
    ✓ advanced__player__network_centrality.csv — 6,194 rows, 1244 players
    ✓ defensive__player__profile.csv — 1,919 rows, 650 players
    ✓ defensive__player__pressures.csv — 5,647 rows, 1167 players

  ── 2022/2023 ──
    ✓ xg__player__totals.csv — 134 rows, 131 players
    ✓ progression__player__profile.csv — 884 rows, 846 players
    ✓ advanced__player__packing.csv — 2,417 rows, 874 players
    ✓ advanced__player__xg_chain.csv — 2,421 rows, 888 players
    ✓ advanced__player__xg_buildup.csv — 2,125 rows, 822 players
    ✓ advanced__player__network_centrality.csv — 2,740 rows, 9

In [23]:
filtered = filter_all(clean, verbose=False)
filtering_summary(filtered)


FILTERING SUMMARY
  xg__player__totals.csv
    599 rows | 528 players | 82 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  progression__player__profile.csv
    1,524 rows | 1304 players | 512 shrinkage-flagged | seasons: ['2021/2022', '2022/2023', '2023/2024']
  advanced__player__packing.csv
    2,086 rows | 1715 players | 384 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__xg_chain.csv
    1,422 rows | 1216 players | 481 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__xg_buildup.csv
    1,280 rows | 1095 players | 424 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__network_centrality.csv
    1,521 rows | 1302 players | 510 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  defensive__player__profile.csv

In [24]:
decayed = apply_decay(filtered, verbose=False)
decay_summary(decayed)


DECAY SUMMARY
  xg__player__totals.csv
    599 rows | 528 players | weights present: [0.8, 0.9, 1.0]
  progression__player__profile.csv
    1,524 rows | 1304 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__packing.csv
    2,086 rows | 1715 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__xg_chain.csv
    1,422 rows | 1216 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__xg_buildup.csv
    1,280 rows | 1095 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__network_centrality.csv
    1,521 rows | 1302 players | weights present: [0.8, 0.9, 1.0]
  defensive__player__profile.csv
    625 rows | 570 players | weights present: [0.8, 0.9, 1.0]
  defensive__player__pressures.csv
    1,378 rows | 1181 players | weights present: [0.8, 0.9, 1.0]


In [25]:
for filename, df in decayed.items():
    metrics_in_file = [
        (k, v["column"]) 
        for k, v in __import__('player_score.player_metrics_config', fromlist=['PLAYER_METRICS']).PLAYER_METRICS.items()
        if v["file"] == filename and v["column"] in df.columns
    ]
    for metric_key, col in metrics_in_file:
        stats = df[col].drop_nulls()
        print(f"{metric_key}: min={stats.min():.3f}, median={stats.median():.3f}, max={stats.max():.3f}, skew={stats.skew():.2f}")

xg_volume: min=0.070, median=0.880, max=23.300, skew=4.98
progressive_passes: min=0.000, median=1.760, max=8.950, skew=1.12
progressive_carries: min=0.000, median=1.155, max=10.840, skew=1.50
packing: min=0.000, median=0.000, max=2.467, skew=1.16
xg_chain: min=0.009, median=0.422, max=1.814, skew=1.07
team_involvement: min=3.175, median=36.130, max=97.559, skew=0.45
xg_buildup: min=0.007, median=0.305, max=1.849, skew=1.29
network_centrality: min=2.241, median=16.362, max=36.332, skew=0.11
defensive_actions: min=27.000, median=63.000, max=639.000, skew=2.83
high_turnovers: min=0.000, median=5.000, max=87.000, skew=3.11
pressure_volume: min=0.796, median=12.177, max=40.275, skew=0.70
pressure_success: min=0.000, median=25.808, max=79.575, skew=0.59


In [26]:
normalized = normalize_all(decayed, verbose=False)
normalization_summary(normalized)


NORMALIZATION SUMMARY
  xg__player__totals.csv: 2 norm columns
    finishing_quality_norm: min=0.115, median=0.500, max=1.000, nulls=0
    xg_volume_norm: min=0.000, median=0.500, max=1.000, nulls=0
  progression__player__profile.csv: 2 norm columns
    progressive_passes_norm: min=0.054, median=0.500, max=1.000, nulls=0
    progressive_carries_norm: min=0.085, median=0.501, max=1.000, nulls=0
  advanced__player__packing.csv: 1 norm columns
    packing_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__xg_chain.csv: 2 norm columns
    xg_chain_norm: min=0.000, median=0.500, max=1.000, nulls=0
    team_involvement_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__xg_buildup.csv: 1 norm columns
    xg_buildup_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__network_centrality.csv: 1 norm columns
    network_centrality_norm: min=0.000, median=0.500, max=1.000, nulls=0
  defensive__player__profile.csv: 2 norm columns
    defensiv

In [27]:
from player_score.steps.shrinkage import apply_shrinkage, shrinkage_summary

LINEUPS_PATH = Path("..") / "data" / "Statsbomb" / "lineups.parquet"

shrunk = apply_shrinkage(normalized, lineups_path=LINEUPS_PATH, verbose=False)
shrinkage_summary(shrunk)


SHRINKAGE SUMMARY
  xg__player__totals.csv
    599 rows | 82 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'W']
    finishing_quality_norm: min=0.115, median=0.522, max=1.000
    xg_volume_norm: min=0.000, median=0.538, max=1.000
  progression__player__profile.csv
    1,524 rows | 512 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    progressive_passes_norm: min=0.054, median=0.511, max=1.000
    progressive_carries_norm: min=0.085, median=0.495, max=1.000
  advanced__player__packing.csv
    2,086 rows | 384 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    packing_norm: min=0.000, median=0.500, max=1.000
  advanced__player__xg_chain.csv
    1,422 rows | 481 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    xg_chain_norm: min=0.000, median=0.542, max=1.000
    team_involvement_norm: min=0.000, median=0.513, max=1.000
  advanced__player__xg_buildup.csv
    1,280 

In [28]:
df = shrunk["advanced__player__network_centrality.csv"]
print(df.filter(pl.col("position_archetype").is_in(["CM", "DM"]))
      .group_by("position_archetype")
      .agg(pl.len().alias("count"))
      .sort("position_archetype"))

shape: (2, 2)
┌────────────────────┬───────┐
│ position_archetype ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ CM                 ┆ 143   │
│ DM                 ┆ 235   │
└────────────────────┴───────┘


In [29]:
segmented = apply_segmentation(shrunk, verbose=False)
segmentation_summary(segmented)


SEGMENTATION SUMMARY

  CB — Centre-Back
    advanced__player__network_centrality.csv: 330 rows
    advanced__player__packing.csv: 445 rows
    advanced__player__xg_buildup.csv: 303 rows
    advanced__player__xg_chain.csv: 310 rows
    defensive__player__pressures.csv: 329 rows
    defensive__player__profile.csv: 75 rows
    progression__player__profile.csv: 122 rows
    xg__player__totals.csv: 50 rows

  FB — Fullback / Wing-Back
    advanced__player__network_centrality.csv: 286 rows
    advanced__player__packing.csv: 446 rows
    advanced__player__xg_buildup.csv: 247 rows
    advanced__player__xg_chain.csv: 267 rows
    defensive__player__pressures.csv: 284 rows
    defensive__player__profile.csv: 150 rows
    progression__player__profile.csv: 104 rows
    xg__player__totals.csv: 53 rows

  DM — Defensive Midfielder
    advanced__player__network_centrality.csv: 235 rows
    advanced__player__packing.csv: 378 rows
    advanced__player__xg_buildup.csv: 204 rows
    advanced__player__x

In [30]:
import steps.scoring
importlib.reload(steps.scoring)
from steps.scoring import * 

scored = build_scores(segmented, verbose=False)
scored = apply_guardian_blend(scored, guardian_weight=0.15, verbose=True)

print(scored[['player', 'team', 'coverage_tier', 
              'composite_score', 'final_score']].head(100).to_string())


  Guardian blend applied:
    In both (blended):      45
    Guardian only (capped): 55
    Model only (unchanged): 118
                                       player                 team coverage_tier composite_score  final_score
rank                                                                                                         
1                        Kylian Mbappé Lottin          Real Madrid             A       67.617055    75.636900
2                      Vitor Machado Ferreira  Paris Saint-Germain             A       54.834517    68.278488
3                 Lamine Yamal Nasraoui Ebana            Barcelona             C       54.220871    68.221411
4                            Fabián Ruiz Peña  Paris Saint-Germain             A       64.447027    66.730007
5                                Granit Xhaka     Bayer Leverkusen             A       68.144491    66.451078
6                             Ousmane Dembélé  Paris Saint-Germain             C       48.087774    64.852665

In [31]:
from rapidfuzz import process, fuzz
import unicodedata

def normalize_name(name: str) -> str:
    normalized = unicodedata.normalize('NFKD', name)
    ascii_name = normalized.encode('ascii', 'ignore').decode('ascii')
    return ascii_name.lower().strip()

def check_guardian_name_mismatches(top_100, scored_df, threshold=75):
    scored_names = scored_df["player"].tolist()
    scored_normalized = {normalize_name(n): n for n in scored_names}
    
    def token_overlap_match(name):
        norm_name = normalize_name(name)
        tokens = set(norm_name.split())
        
        if len(tokens) >= 3:
            best_score = 0
            best_match = None
            for norm_candidate, orig_candidate in scored_normalized.items():
                candidate_tokens = set(norm_candidate.split())
                overlap = tokens & candidate_tokens
                if len(overlap) >= 2:
                    score = (len(overlap) / max(len(tokens), len(candidate_tokens))) * 100
                    if score > best_score:
                        best_score = score
                        best_match = orig_candidate
            if best_match:
                return best_match, best_score
        
        result = process.extractOne(
            norm_name,
            list(scored_normalized.keys()),
            scorer=fuzz.token_sort_ratio
        )
        if result:
            return scored_normalized[result[0]], result[1]
        return None, 0

    print(f"GUARDIAN NAME MISMATCHES (threshold={threshold}):")
    print("-" * 70)
    
    no_match = []
    mismatches = []
    exact = []

    for entry in top_100:
        name = entry['player']
        
        # Exact match
        if name in scored_names:
            exact.append(name)
            continue
        
        # Normalized exact match
        if normalize_name(name) in scored_normalized:
            exact.append(name)
            continue
        
        matched, score = token_overlap_match(name)
        
        if matched and score >= threshold:
            mismatches.append((score, entry['rank'], name, matched))
        else:
            no_match.append((entry['rank'], name))

    print(f"\nEXACT MATCHES ({len(exact)}) — no action needed")
    
    print(f"\nPOTENTIAL FIXES ({len(mismatches)}):")
    for score, rank, name, matched in sorted(mismatches, key=lambda x: -x[0]):
        print(f"  {score:3}  #{rank} {name!r}")
        print(f"       → {matched!r}")
        print()
    
    print(f"\nNO MATCH FOUND ({len(no_match)}):")
    for rank, name in sorted(no_match):
        print(f"  #{rank} {name!r}")

check_guardian_name_mismatches(TOP_100, scored, threshold=75)

NameError: name 'TOP_100' is not defined